## Atelier 0 — Mise en route (5 min)
Les **paramètres** sont regroupés en tête de notebook : c'est une règle d'auditabilité (chemins et seuils ne se cachent pas au milieu du code).

In [ ]:
# --- PARAMÈTRES (toujours en tête du notebook) ---------------------------
CHEMIN_DONNEES = "portefeuille_vie.csv"
DATE_REF = "2025-01-01"     # début de la période d'observation (rachat_12m = rachat sur l'exercice 2025)
SEUIL_EFFECTIF = 100        # en dessous, un taux par segment est jugé peu fiable

# --- IMPORTS --------------------------------------------------------------
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__)


# --- OUTIL D'AUTO-CONTRÔLE (ne pas modifier) ------------------------------
def verifier(libelle, test):
    """Affiche ✅ ou ❌ selon le résultat de `test` (une fonction sans argument)."""
    try:
        ok = bool(test())
    except Exception:
        ok = False
    print(("✅ " if ok else "❌ ") + libelle + ("" if ok else "   → à compléter ou à revoir"))

---
# Atelier B — Charger et contrôler le portefeuille (≈ 28 min)
Objectif : **cartographier les défauts avant de calculer quoi que ce soit.** Le jeu de données contient des défauts volontaires.

### B1 ·  Charger et lire le schéma
Chargez le CSV en convertissant `date_effet` en vraie date, puis vérifiez dimensions et types : *10 secondes qui évitent des heures de débogage.*
<details><summary>💡 Indice</summary><code>pd.read_csv(chemin, parse_dates=["nom_colonne"])</code>. Les dimensions s'obtiennent avec <code>.shape</code> (lignes, colonnes).</details>

In [ ]:
df_brut = pd.read_csv(CHEMIN_DONNEES, parse_dates=[...])

print("Dimensions (lignes, colonnes) :", df_brut....)
print(df_brut.dtypes)
df_brut.head()

In [ ]:
verifier("12 colonnes", lambda: df_brut.shape[1] == 12)
verifier("date_effet est bien une date", lambda: str(df_brut.date_effet.dtype).startswith("datetime"))

### B2 ·  Cartographier les trois familles de défauts
**Manquants, doublons, valeurs hors domaine.** Comptez-les, sans rien corriger encore.
- (a) Part de valeurs manquantes par colonne, de la plus touchée à la moins touchée
- (b) Nombre de contrats en double (clé `id_contrat`)
- (c) Âges hors de [18 ; 100] et encours négatifs ou nuls
<details><summary>💡 Indice</summary>(a) <code>df.isna().mean()</code> donne la part de manquants ; <code>.sort_values(ascending=False)</code> trie. (b) <code>df.duplicated("id_contrat").sum()</code>. (c) Un masque booléen : <code>(df.age &lt; 18) | (df.age &gt; 100)</code>.</details>

In [ ]:
# (a) Part de manquants par colonne
manquants = df_brut.isna().mean().sort_values(ascending=...)
print((manquants * 100).round(1))

# (b) Doublons sur la clé id_contrat
nb_doublons = df_brut.duplicated(...).sum()

# (c) Valeurs hors domaine
nb_age_hors_domaine = ((df_brut.age < 18) | (df_brut.age > ...)).sum()
nb_encours_invalide = (df_brut.encours ... 0).sum()

print(f"\nDoublons : {nb_doublons} | âges hors domaine : {nb_age_hors_domaine} | encours <= 0 : {nb_encours_invalide}")

verifier("30 doublons détectés", lambda: nb_doublons == 30)
verifier("6 âges hors domaine", lambda: nb_age_hors_domaine == 6)
verifier("8 encours invalides", lambda: nb_encours_invalide == 8)
verifier("rachat_hist ≈ 41 % de manquants", lambda: 0.40 < manquants["rachat_hist"] < 0.42)

### B3 ·  Le manquant n'est pas neutre
41 % des contrats n'ont pas de `rachat_hist`. **Défaillance du système ou donnée structurellement absente ?** Les deux situations ont des conséquences opposées sur le modèle. Calculez la part de manquants **par génération** pour trancher.
<details><summary>💡 Indice</summary><code>df.rachat_hist.isna()</code> donne un booléen par contrat ; <code>.groupby(df.generation).mean()</code> en fait une part par génération.</details>

In [ ]:
part_manquants_gen = df.rachat_hist.isna().groupby(df....).mean()
print((part_manquants_gen * 100).round(1))

verifier("100 % de manquants pour la génération 2024", lambda: part_manquants_gen[2024] == 1.0)
verifier("moins de 15 % de manquants pour 2015-2022", lambda: part_manquants_gen.loc[2015:2022].max() < 0.15)

**Votre lecture métier** *(2 phrases)* : que révèle ce tableau ? Quel traitement envisagez-vous pour chacune des deux populations de manquants ?

*Votre réponse :*

### B4 ·  Le piège du filtre « contrats en cours » (et la fuite de cible)
Un collègue propose de calculer le taux de rachat sur *« les contrats en cours »* (`statut == "en cours"`). Testez. Puis croisez `statut` et `rachat_12m` pour comprendre.
<details><summary>💡 Indice</summary><code>pd.crosstab(df.colonne_lignes, df.colonne_colonnes)</code> compte les effectifs de chaque combinaison.</details>

In [ ]:
actifs = df[df.statut == "en cours"]

taux_biaise = actifs.....mean()          # taux de rachat calculé sur ces contrats
tableau = pd.crosstab(df...., df....)     # statut en lignes, rachat_12m en colonnes

print("Taux de rachat sur les contrats 'en cours' :", taux_biaise)
print(tableau)

verifier("le taux « biaisé » vaut exactement 0", lambda: taux_biaise == 0)

**Votre lecture** *(2-3 phrases)* : pourquoi 0 % ? Que dit le tableau croisé ? Quelle conséquence pour une future modélisation ?

*Votre réponse :*